In [61]:
import pandas as pd
import os

def process_csv_files(input_directory, output_directory):
    for root, _, files in os.walk(input_directory):
        for file in files:
            if file.endswith('.csv'):
                csv_file = os.path.join(root, file)
                
                # Read the CSV file, ensure 'Time' column is read as string
                df = pd.read_csv(csv_file, dtype=str, header=None)

                # Extract AHU name from the folder structure
                folder_parts = os.path.normpath(root).split(os.sep)
                ahu_name = folder_parts[-1] if 'AHU-' in folder_parts[-1] else 'Unknown AHU'

                # Use the file name (excluding the extension) as the parameter name
                parameter = os.path.splitext(file)[0]

                # Handle different cases based on the number of columns
                if len(df.columns) == 4:
                    df.columns = ['Description', 'Parameter', 'Time', 'Value']
                    df = df[['Time', 'Value']]
                elif len(df.columns) == 3:
                    df.columns = ['Description', 'Time', 'Value']
                    df = df[['Time', 'Value']]
                elif len(df.columns) == 2:
                    df.columns = ['Time', 'Value']
                else:
                    print(f"Unexpected structure in {csv_file}: {df.columns}")
                    continue

                # Add AHU name column and rename the 'Value' column to the parameter
                df.insert(0, 'AHU name', ahu_name)
                df.rename(columns={'Value': parameter}, inplace=True)

                # Debug: Check and print the presence of '00' timestamp
                if '2024-05-01 00' in df['Time'].values:
                    print(f"'00' timestamp found in {file}")
                else:
                    print(f"'00' timestamp missing in {file}")

                # Debug: Print the first few rows of the processed CSV
                print(f"First few rows of {file} after processing:")
                print(df.head())

                # Save the updated dataframe
                relative_path = os.path.relpath(root, input_directory)
                output_dir = os.path.join(output_directory, relative_path)
                os.makedirs(output_dir, exist_ok=True)
                processed_csv_file = os.path.join(output_dir, file)
                df.to_csv(processed_csv_file, index=False)
                print(f"Processed {csv_file} to {processed_csv_file}")

# Example usage for processing
processed_input_folder = "/Users/seunghyeon/Desktop/fwd/example1"
processed_output_folder = "/Users/seunghyeon/Desktop/fwd/example_processed"

process_csv_files(processed_input_folder, processed_output_folder)

'00' timestamp found in Total heating pump.csv
First few rows of Total heating pump.csv after processing:
      AHU name           Time Total heating pump
0  Unknown AHU  2024-05-01 00                  0
1  Unknown AHU  2024-05-01 01                  0
2  Unknown AHU  2024-05-01 02                  0
3  Unknown AHU  2024-05-01 03                  0
4  Unknown AHU  2024-05-01 04                  0
Processed /Users/seunghyeon/Desktop/fwd/example1/office 24. May/Total heating pump.csv to /Users/seunghyeon/Desktop/fwd/example_processed/office 24. May/Total heating pump.csv
'00' timestamp found in Heating supply temperature 1.csv
First few rows of Heating supply temperature 1.csv after processing:
      AHU name           Time Heating supply temperature 1
0  Unknown AHU  2024-05-01 00                        22.36
1  Unknown AHU  2024-05-01 01                        22.37
2  Unknown AHU  2024-05-01 02                        22.37
3  Unknown AHU  2024-05-01 03                        22.38
4  

In [72]:
import pandas as pd
import glob
import os

# Base directory containing the datasets
base_dir = "/Users/seunghyeon/Desktop/fwd/example_processed"

# Function to standardize the Time column format
def standardize_time_format(df):
    df['Time'] = pd.to_datetime(df['Time'], format='%Y-%m-%d %H')
    return df

# Function to load shared feature files for a given month directory
def load_shared_features_for_month(month_dir):
    shared_features_keywords = [
        'Total heating pump', 'Heating supply temperature 1', 'Heating pump 2',
        'Heating supply temperature 2', 'Heating pump 1', 'Backup heating pump'
    ]
    shared_files = [file for file in glob.glob(f"{month_dir}/*.csv") if any(keyword in file for keyword in shared_features_keywords)]
    shared_dfs = [standardize_time_format(pd.read_csv(file)) for file in shared_files]
    
    # Merge all shared feature dataframes on 'Time' column
    combined_shared_df = shared_dfs[0]
    for df in shared_dfs[1:]:
        combined_shared_df = pd.merge(combined_shared_df, df, on="Time", how="outer")
    
    return combined_shared_df

# Function to load and merge data for each AHU in a given month directory with shared features
def load_and_merge_ahu_data_for_month(ahu_id, month_dir, shared_features):
    # Find all distinct feature files for the given AHU
    ahu_files = glob.glob(f"{month_dir}/{ahu_id}/*.csv")
    
    # Load distinct feature dataframes
    distinct_dfs = [standardize_time_format(pd.read_csv(file)) for file in ahu_files]
    
    # Merge all distinct feature dataframes on 'Time' column with suffixes to handle duplicates
    merged_df = distinct_dfs[0]
    for df in distinct_dfs[1:]:
        merged_df = pd.merge(merged_df, df, on=["Time"], how="outer", suffixes=('', '_dup'))
    
    # Drop duplicate columns
    merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]
    
    # Add AHU name to the merged dataframe
    merged_df['AHU name'] = ahu_id
    
    # Merge shared features based on 'Time' with AHU data
    merged_df = pd.merge(merged_df, shared_features, on="Time", how="left")
    
    # Drop unnecessary AHU name columns if they exist
    merged_df = merged_df.drop(columns=[col for col in merged_df.columns if 'AHU name_' in col], errors='ignore')
    
    return merged_df

# Get a list of all month directories
month_dirs = [d for d in glob.glob(f"{base_dir}/*") if os.path.isdir(d)]

# Process each month directory
all_combined_data = []
for month_dir in month_dirs:
    # Load shared features for the current month
    shared_features = load_shared_features_for_month(month_dir)
    
    # Find all unique AHU IDs (directories) in the current month directory
    ahu_ids = set([os.path.basename(file).split('/')[-1] for file in glob.glob(f"{month_dir}/AHU-*")])
    
    # Load and merge data for all AHUs in the current month with shared features
    month_combined_data = pd.concat([load_and_merge_ahu_data_for_month(ahu_id, month_dir, shared_features) for ahu_id in ahu_ids], ignore_index=True)
    
    all_combined_data.append(month_combined_data)

# Combine data from all months
final_combined_data = pd.concat(all_combined_data, ignore_index=True)

# Function to clean duplicate columns
def clean_and_verify_columns(df):
    # Identify columns with '_dup' suffix
    columns_to_drop = [col for col in df.columns if col.endswith('_dup')]
    
    # Drop duplicate columns
    df_cleaned = df.drop(columns=columns_to_drop, errors='ignore')
    
    # Fill missing AHU name if there's a duplicate column
    if 'AHU name_dup' in df_cleaned.columns:
        df_cleaned['AHU name'] = df_cleaned['AHU name'].fillna(df_cleaned['AHU name_dup'])
        df_cleaned = df_cleaned.drop(columns=['AHU name_dup'])
    
    return df_cleaned

# Clean the dataset by removing duplicate columns and verifying
cleaned_final_combined_data = clean_and_verify_columns(final_combined_data)

# Convert 'Time' column to datetime format and sort
cleaned_final_combined_data['Time'] = pd.to_datetime(cleaned_final_combined_data['Time'], format='%Y-%m-%d %H')
cleaned_final_combined_data = cleaned_final_combined_data.sort_values(by=['AHU name', 'Time'])

# Reset index for better display
cleaned_final_combined_data.reset_index(drop=True, inplace=True)

# Save the final cleaned combined dataset as a CSV file
output_file_path = '/Users/seunghyeon/Desktop/fwd/final_cleaned_combined_data.csv'
cleaned_final_combined_data.to_csv(output_file_path, index=False)

print(f"Final merged dataset saved to {output_file_path}")

# Display the structure of the cleaned dataset
import ace_tools as tools; tools.display_dataframe_to_user(name="Final Cleaned Combined AHU Data", dataframe=cleaned_final_combined_data)
cleaned_final_combined_data.info(), cleaned_final_combined_data.head()


<ipython-input-72-2e5d65a1a0f3>:25: FutureWarning: Passing 'suffixes' which cause duplicate columns {'AHU name_x'} in the result is deprecated and will raise a MergeError in a future version.
  combined_shared_df = pd.merge(combined_shared_df, df, on="Time", how="outer")
<ipython-input-72-2e5d65a1a0f3>:25: FutureWarning: Passing 'suffixes' which cause duplicate columns {'AHU name_x'} in the result is deprecated and will raise a MergeError in a future version.
  combined_shared_df = pd.merge(combined_shared_df, df, on="Time", how="outer")
<ipython-input-72-2e5d65a1a0f3>:25: FutureWarning: Passing 'suffixes' which cause duplicate columns {'AHU name_x'} in the result is deprecated and will raise a MergeError in a future version.
  combined_shared_df = pd.merge(combined_shared_df, df, on="Time", how="outer")
<ipython-input-72-2e5d65a1a0f3>:25: FutureWarning: Passing 'suffixes' which cause duplicate columns {'AHU name_x'} in the result is deprecated and will raise a MergeError in a future v

Final merged dataset saved to /Users/seunghyeon/Desktop/fwd/final_cleaned_combined_data.csv


ModuleNotFoundError: No module named 'ace_tools'